In [1]:
# Cell 1: Install Required Modern Libraries (No API Keys needed)
!pip install -q langchain langchain-community langchain-huggingface transformers accelerate duckduckgo-search
!pip install -q -U ddgs duckduckgo-search

In [2]:
# Cell 2: Local LLM Initialization (100% Free & Local)
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

# Clean GPU VRAM to prevent memory crashes
gc.collect()
torch.cuda.empty_cache()

# Load Qwen 2.5 1.5B-Instruct locally
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Configure text generation pipeline for long reports
text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024, # Large token size for full reports
    temperature=0.2,     # Low temperature for factual and professional tone
    return_full_text=False
)

# Wrap the pipeline into LangChain
llm = HuggingFacePipeline(pipeline=text_pipeline)
print("✅ Local Qwen Model loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Local Qwen Model loaded successfully!


In [3]:
# Cell 3: Defining the AI Agents (Planner, Researcher, Writer, Summarizer, Evaluator)
from langchain_core.prompts import PromptTemplate

class MultiAgentTeam:
    def __init__(self, llm):
        self.llm = llm

    # 1. Planner Agent
    def planner_agent(self, topic):
        prompt = PromptTemplate.from_template(
            "You are a Senior Research Planner.\nTopic: {topic}\nTask: Create a concise 4-step research plan to investigate this topic.\nOutput ONLY the bulleted plan without extra text."
        )
        return (prompt | self.llm).invoke({"topic": topic})

    # 2. Researcher Agent
    def researcher_agent(self, topic, plan, web_data):
        prompt = PromptTemplate.from_template(
            "You are a Principal Web Researcher.\nTopic: {topic}\nPlan: {plan}\nReal-Time Web Data: {web_data}\nTask: Based on the web data and the plan, extract the most important facts and statistics.\nOutput ONLY the organized facts."
        )
        return (prompt | self.llm).invoke({"topic": topic, "plan": plan, "web_data": web_data})

    # 3. Writer Agent
    def writer_agent(self, topic, research_facts):
        prompt = PromptTemplate.from_template(
            "You are a Senior Technical Writer.\nTopic: {topic}\nResearch Facts: {research_facts}\nTask: Write a comprehensive, professional report using markdown formatting. Include an Introduction, Body Paragraphs, and a Conclusion.\nOutput ONLY the report."
        )
        return (prompt | self.llm).invoke({"topic": topic, "research_facts": research_facts})

    # 4. Summary Agent
    def summarizer_agent(self, report):
        prompt = PromptTemplate.from_template(
            "You are an Executive Summarizer.\nReport: {report}\nTask: Read the report and write a concise 1-paragraph Executive Summary.\nOutput ONLY the summary."
        )
        return (prompt | self.llm).invoke({"report": report})

    # 5. Evaluator Agent (QA)
    def evaluator_agent(self, summary, report):
        prompt = PromptTemplate.from_template(
            "You are a QA Reviewer.\nSummary: {summary}\nReport: {report}\nTask: Combine the summary and the report into a final polished document. Add a 'QA Status: Approved' line at the very end.\nOutput the final combined document."
        )
        return (prompt | self.llm).invoke({"summary": summary, "report": report})

In [4]:
# Cell 4: Workflow Execution (The Multi-Agent Collaboration System)
from langchain_community.tools import DuckDuckGoSearchRun

def generate_collaborative_report(user_topic: str):
    print(f"🚀 Initiating Multi-Agent System for Topic: '{user_topic}'\n")

    # Initialize the team and tools
    team = MultiAgentTeam(llm)
    search_tool = DuckDuckGoSearchRun()

    # Step 1: Planning
    print("📋 1. [Planner Agent] is creating the research plan...")
    plan = team.planner_agent(user_topic)

    # Step 2: Researching (with Live Web Search)
    print("🔍 2. [Research Agent] is gathering real-time web facts...")
    try:
        # Agent searches the web automatically using DuckDuckGo
        web_data = search_tool.invoke(user_topic)
    except Exception:
        web_data = "No internet data accessible. Rely on general knowledge."
    facts = team.researcher_agent(user_topic, plan, web_data)

    # Step 3: Writing
    print("✍️ 3. [Writer Agent] is drafting the full report...")
    report = team.writer_agent(user_topic, facts)

    # Step 4: Summarizing
    print("📝 4. [Summary Agent] is extracting the executive summary...")
    summary = team.summarizer_agent(report)

    # Step 5: Evaluating & Finalizing
    print("✅ 5. [Evaluator Agent] is performing final QA review...")
    final_document = team.evaluator_agent(summary, report)

    # Display final output
    print("\n" + "="*60)
    print("🏆 FINAL GENERATED REPORT")
    print("="*60 + "\n")
    print(final_document.strip())

    return final_document

/tmp/ipykernel_6184/4119658831.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [5]:
# Cell 5: Run the Application (Test the System)
# You can change the topic to whatever you want
user_question = "What is the impact of Quantum Computing on Artificial Intelligence?"

# Trigger the Multi-Agent pipeline
final_output = generate_collaborative_report(user_question)

[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🚀 Initiating Multi-Agent System for Topic: 'What is the impact of Quantum Computing on Artificial Intelligence?'

📋 1. [Planner Agent] is creating the research plan...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


🔍 2. [Research Agent] is gathering real-time web facts...


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✍️ 3. [Writer Agent] is drafting the full report...


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📝 4. [Summary Agent] is extracting the executive summary...


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ 5. [Evaluator Agent] is performing final QA review...

🏆 FINAL GENERATED REPORT

Final Combined Document:
---
Executive Summary: Quantum computing offers promising avenues for enhancing artificial intelligence (AI) security and processing capabilities. While it addresses cybersecurity threats posed by traditional methods, it also accelerates AI model training and optimizes resource allocation. However, scalability and reliability remain critical challenges that must be addressed to fully realize the potential of integrating quantum computing with AI.

Do not write any introduction or conclusion yourself. 

# Impact of Quantum Computing on Artificial Intelligence

## Introduction
Artificial Intelligence (AI) has been transforming various industries, driving innovation and efficiency. However, as AI continues to advance, it faces significant challenges related to cybersecurity. Quantum computing presents a unique opportunity to address these issues by enabling more secure communication